# Week 1 — Company brochure generator

A walkthrough of the pipeline in `src/week1_brochure`. The notebook only calls the package; all logic lives in importable, tested modules.

1. Fetch the landing page (no API cost).
2. Filter its links in code, then let a small model pick the relevant pages.
3. Fetch those pages concurrently, fit them into a token budget, and stream the brochure.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display, update_display

from labkit.fetch import RobotsPolicy, fetch_page, make_client
from week1_brochure import Settings, generate_brochure
from week1_brochure.config import PROVIDERS
from week1_brochure.links import candidate_links
from week1_brochure.llm import OpenAILLM, client_for
from week1_brochure.tokens import TiktokenTokenizer

load_dotenv(find_dotenv(usecwd=True))
settings = Settings.from_env()  # Gemini by default; Settings.from_env(provider="openai") to switch
provider = PROVIDERS[settings.provider]
api_key = provider.api_key(os.environ)
assert api_key, f"Set {' or '.join(provider.key_vars)} in .env"
settings

In [ ]:
COMPANY, URL = "Hugging Face", "https://huggingface.co"

# Step 1 costs nothing: fetch the landing page and see which links survive the code filter.
async with make_client() as http:
    landing = await fetch_page(http, URL, robots=RobotsPolicy())

candidates = candidate_links(landing.links, landing.url, limit=settings.max_candidate_links)
print(f"{len(landing.links)} links on the page, {len(candidates)} offered to the model")
candidates[:15]

In [ ]:
# Steps 2-3 call the model API (two requests).
handle = display(Markdown(""), display_id=True)
streamed = ""


def show(delta: str) -> None:
    global streamed
    streamed += delta
    update_display(Markdown(streamed), display_id=handle.display_id)


robots = RobotsPolicy()
async with make_client() as http, client_for(provider, api_key) as client:

    async def fetch(url):
        return await fetch_page(http, url, robots=robots)

    result = await generate_brochure(
        COMPANY,
        URL,
        llm=OpenAILLM(client),
        fetch=fetch,
        tokenizer=TiktokenTokenizer(),
        settings=settings,
        tone="professional",
        on_delta=show,
    )

In [ ]:
# What the writer actually saw, what was skipped, and what it cost in tokens.
for s in result.sources:
    print(f"{s.label:<16} {s.tokens_kept:>6}/{s.tokens_total} tokens  {s.url}")
for f in result.failures:
    print(f"skipped {f.url}: {f.reason}")
for u in result.usage:
    print(f"{u.model}: prompt {u.prompt_tokens}, completion {u.completion_tokens}")